# 02 — Entrenamiento CNN y Evaluación
**EnergyFingerprint Pipeline** | Notebook 2 de 3

Entrena la EnergySignalCNN con validación cruzada, evalúa métricas (AUC, F1),
compara 9ch vs 11ch, y ejecuta ablación de canales.

**Entrada**: `data/<gen>_tensors_<N>ch.npz` (generado por notebook 01)
**Salida**: Modelos entrenados, figuras ROC, resultados de ablación

## Configuración

In [ ]:
# ══════════════════════════════════════════════════
# CONFIGURAR AQUÍ
# ══════════════════════════════════════════════════
GENE        = "scn1a"
SUBSET      = "likely"
N_CHANNELS  = 11         # canales del tensor a cargar
N_FOLDS     = 5          # folds de cross-validation
EPOCHS      = 100        # épocas máximas (early stopping activo)
PATIENCE    = 12         # paciencia early stopping
BATCH_SIZE  = 32
LR          = 1e-3
DROPOUT     = 0.2
SEED        = 42
# ══════════════════════════════════════════════════

## Imports

In [ ]:
import sys, os, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from energyfingerprint.model import (
    EnergySignalCNN, VariantDataset,
    EarlyStopping, create_weighted_sampler, GradCAM1D
)

STUDY_DIR = os.path.join(PROJECT_ROOT, "studies", GENE)
DATA_DIR  = os.path.join(STUDY_DIR, "data")
FIG_DIR   = os.path.join(STUDY_DIR, "figures")
MODEL_DIR = os.path.join(STUDY_DIR, "models")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Device
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Gen: {GENE.upper()} | Canales: {N_CHANNELS} | Device: {device}")

## 1. Cargar tensores

In [ ]:
tensor_file = os.path.join(DATA_DIR,
    f"{GENE}_tensors_{N_CHANNELS}ch{'_' + SUBSET if SUBSET != 'all' else ''}.npz")

data = np.load(tensor_file, allow_pickle=True)
tensors = data["tensors"]
labels  = data["labels"]
uids    = data["variant_ids"]

print(f"Tensores: {tensors.shape}")
print(f"Labels: {(labels==1).sum()} P / {(labels==0).sum()} B")
print(f"Archivo: {tensor_file}")

## 2. Funciones de entrenamiento

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, n = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n

def eval_model(model, loader, device):
    model.eval()
    probs_all, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            probs = torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().numpy()
            probs_all.extend(probs)
            labels_all.extend(y.numpy())
    l, p = np.array(labels_all), np.array(probs_all)
    preds = (p >= 0.5).astype(int)
    auc = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    return {"auc": auc, "f1": f1_score(l, preds, zero_division=0),
            "acc": accuracy_score(l, preds), "probs": p, "labels": l}

## 3. Entrenamiento con K-Fold Cross-Validation

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
all_probs, all_labels = [], []
best_models = []

print(f"Entrenando {N_FOLDS}-fold CV ({EPOCHS} epochs max, patience={PATIENCE})\n")

for fold_i, (train_idx, val_idx) in enumerate(skf.split(tensors, labels)):
    t0 = time.time()
    tr_t, tr_l = tensors[train_idx], labels[train_idx]
    va_t, va_l = tensors[val_idx], labels[val_idx]

    tr_ds = VariantDataset(tr_t, tr_l, augment=True)
    va_ds = VariantDataset(va_t, va_l, augment=False)
    sampler = create_weighted_sampler(tr_l)
    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    va_ld = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = EnergySignalCNN(n_channels=N_CHANNELS, n_classes=2,
                            dropout=DROPOUT, n_heads=4).to(device)
    cc = np.bincount(tr_l)
    cw = torch.FloatTensor([len(tr_l) / (2 * c) for c in cc]).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                      factor=0.5, patience=7)
    es = EarlyStopping(patience=PATIENCE)

    best_auc, best_state = 0, None
    history = {"train_loss": [], "val_auc": []}

    for ep in range(EPOCHS):
        loss = train_epoch(model, tr_ld, criterion, optimizer, device)
        m = eval_model(model, va_ld, device)
        scheduler.step(1 - m["auc"])
        history["train_loss"].append(loss)
        history["val_auc"].append(m["auc"])
        if m["auc"] > best_auc:
            best_auc = m["auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if es(1 - m["auc"]):
            break

    model.load_state_dict(best_state)
    m = eval_model(model, va_ld, device)
    fold_results.append(m)
    all_probs.extend(m["probs"])
    all_labels.extend(m["labels"])
    best_models.append(best_state)

    elapsed = time.time() - t0
    print(f"  Fold {fold_i+1}: AUC={m['auc']:.4f}  F1={m['f1']:.4f}  "
          f"Acc={m['acc']:.4f}  ({ep+1} epochs, {elapsed:.0f}s)")

auc_mean = np.mean([r["auc"] for r in fold_results])
auc_std  = np.std([r["auc"] for r in fold_results])
f1_mean  = np.mean([r["f1"] for r in fold_results])
print(f"\n>>> AUC: {auc_mean:.4f} ± {auc_std:.4f}  |  F1: {f1_mean:.4f}")

## 4. Curva ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{GENE.upper()} — EnergySignalCNN {N_CHANNELS}ch ({SUBSET})", fontweight="bold")

# Per-fold ROC
ax = axes[0]
for i, r in enumerate(fold_results):
    fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
    ax.plot(fpr, tpr, lw=1.5, alpha=0.7, label=f"Fold {i+1} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.4)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title(f"ROC por fold ({N_FOLDS}-fold CV)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# Aggregated ROC
ax = axes[1]
all_l = np.array(all_labels)
all_p = np.array(all_probs)
fpr, tpr, _ = roc_curve(all_l, all_p)
ax.plot(fpr, tpr, color="#2196F3", lw=2.5, label=f"AUC={auc_mean:.4f} ± {auc_std:.4f}")
ax.fill_between(fpr, 0, tpr, alpha=0.1, color="#2196F3")
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.4)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC agregada")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"cnn_{N_CHANNELS}ch_roc.png"), dpi=150)
plt.show()

## 5. Guardar modelos

In [ ]:
model_subdir = os.path.join(MODEL_DIR, f"{N_CHANNELS}ch_{SUBSET}")
os.makedirs(model_subdir, exist_ok=True)

for i, state in enumerate(best_models):
    torch.save(state, os.path.join(model_subdir, f"fold_{i}.pt"))

# Guardar resumen
summary = {
    "gene": GENE,
    "n_channels": N_CHANNELS,
    "subset": SUBSET,
    "n_folds": N_FOLDS,
    "auc_mean": float(auc_mean),
    "auc_std": float(auc_std),
    "f1_mean": float(f1_mean),
    "folds": [{"auc": float(r["auc"]), "f1": float(r["f1"]),
               "acc": float(r["acc"])} for r in fold_results],
}
with open(os.path.join(model_subdir, "results.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"Modelos guardados en: {model_subdir}")
print(f"AUC: {auc_mean:.4f} ± {auc_std:.4f}")

## 6. Ablación de canales (Leave-One-Out)

Evalúa la contribución de cada canal eliminándolo y midiendo Δ AUC.
También evalúa grupos funcionales (solo energía, solo contexto, solo ESM).

In [ ]:
# Configuración de ablación
ABL_FOLDS  = 3
ABL_EPOCHS = 60

CHANNEL_NAMES = ["ΔG_SL_wt", "ΔG_Tu_wt", "ΔG_SL_mut", "ΔG_Tu_mut",
                 "ΔΔG_SL", "ΔΔG_Tu", "GC", "Purine", "CodonPos",
                 "SecStruct", "ESM_LLR"][:N_CHANNELS]

# Baseline: todos los canales
def quick_cv(t, l, n_ch, n_folds=3, epochs=60, seed=42):
    """CV rápida para ablación."""
    np.random.seed(seed); torch.manual_seed(seed)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    aucs = []
    for tri, vi in skf.split(t, l):
        tr_ds = VariantDataset(t[tri], l[tri], augment=True)
        va_ds = VariantDataset(t[vi], l[vi], augment=False)
        sampler = create_weighted_sampler(l[tri])
        tr_ld = DataLoader(tr_ds, batch_size=32, sampler=sampler, num_workers=0)
        va_ld = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=0)

        model = EnergySignalCNN(n_channels=n_ch, n_classes=2, dropout=0.2, n_heads=4).to(device)
        cc = np.bincount(l[tri])
        cw = torch.FloatTensor([len(l[tri]) / (2*c) for c in cc]).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        es = EarlyStopping(patience=10)
        best_auc, best_st = 0, None
        for ep in range(epochs):
            train_epoch(model, tr_ld, criterion, optimizer, device)
            m = eval_model(model, va_ld, device)
            if m["auc"] > best_auc:
                best_auc = m["auc"]
                best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if es(1 - m["auc"]): break
        aucs.append(best_auc)
    return np.mean(aucs), np.std(aucs)

print("Ablación de canales (leave-one-out)...")
print(f"Config: {ABL_FOLDS}-fold CV, {ABL_EPOCHS} epochs\n")

baseline_auc, baseline_std = quick_cv(tensors, labels, N_CHANNELS,
                                       ABL_FOLDS, ABL_EPOCHS, SEED)
print(f"Baseline ({N_CHANNELS}ch): AUC = {baseline_auc:.4f} ± {baseline_std:.4f}\n")

ablation = {}
for ch_i in range(N_CHANNELS):
    t_abl = np.delete(tensors, ch_i, axis=2)
    auc_abl, std_abl = quick_cv(t_abl, labels, N_CHANNELS - 1,
                                 ABL_FOLDS, ABL_EPOCHS, SEED)
    delta = baseline_auc - auc_abl
    ablation[CHANNEL_NAMES[ch_i]] = {
        "auc": auc_abl, "std": std_abl, "delta": delta
    }
    print(f"  Sin {CHANNEL_NAMES[ch_i]:12s}: AUC={auc_abl:.4f}  Δ={delta:+.4f}")

### Ablación por grupos funcionales

In [ ]:
groups = {
    "Solo energía (C1-C6)": list(range(6)),
    "Solo contexto (C7-C9)": list(range(6, 9)),
}
if N_CHANNELS >= 11:
    groups["Solo ESM (C11)"] = [10]
    groups["Energía + ESM (C1-C6,C11)"] = list(range(6)) + [10]

group_results = {}
for name, ch_list in groups.items():
    t_grp = tensors[:, :, ch_list]
    auc_g, std_g = quick_cv(t_grp, labels, len(ch_list), ABL_FOLDS, ABL_EPOCHS, SEED)
    group_results[name] = {"auc": auc_g, "std": std_g}
    print(f"  {name:30s}: AUC={auc_g:.4f} ± {std_g:.4f}")

### Figura de ablación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"{GENE.upper()} — Ablación de canales ({N_CHANNELS}ch)", fontweight="bold")

# Leave-one-out
ax = axes[0]
names = list(ablation.keys())
deltas = [ablation[n]["delta"] for n in names]
colors = ["#E91E63" if d > 0.01 else "#2196F3" if d > 0 else "#4CAF50" for d in deltas]
bars = ax.barh(range(len(names)), deltas, color=colors, alpha=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Δ AUC (baseline - sin canal)")
ax.set_title("Leave-One-Out")
ax.axvline(x=0, color="gray", ls="--", alpha=0.5)
ax.grid(True, alpha=0.2, axis="x")
for b, d in zip(bars, deltas):
    ax.text(d + 0.002 if d >= 0 else d - 0.002, b.get_y() + b.get_height()/2,
            f"{d:+.4f}", va="center", ha="left" if d >= 0 else "right", fontsize=8)

# Groups
ax = axes[1]
g_names = list(group_results.keys()) + [f"Full ({N_CHANNELS}ch)"]
g_aucs  = [group_results[n]["auc"] for n in group_results] + [baseline_auc]
g_cols  = ["#FF9800"] * len(group_results) + ["#2196F3"]
bars = ax.barh(range(len(g_names)), g_aucs, color=g_cols, alpha=0.8)
ax.set_yticks(range(len(g_names)))
ax.set_yticklabels(g_names, fontsize=9)
ax.set_xlabel("AUC")
ax.set_title("Grupos funcionales")
ax.set_xlim(0.5, 1.0)
ax.grid(True, alpha=0.2, axis="x")
for b, v in zip(bars, g_aucs):
    ax.text(v + 0.005, b.get_y() + b.get_height()/2,
            f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"cnn_ablation_{N_CHANNELS}ch.png"), dpi=150)
plt.show()

In [ ]:
ablation_results = {
    "baseline": {"auc": float(baseline_auc), "std": float(baseline_std)},
    "leave_one_out": {k: {kk: float(vv) for kk, vv in v.items()}
                      for k, v in ablation.items()},
    "groups": {k: {kk: float(vv) for kk, vv in v.items()}
               for k, v in group_results.items()},
}
abl_path = os.path.join(DATA_DIR, f"ablation_{N_CHANNELS}ch.json")
with open(abl_path, "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"Resultados de ablación guardados: {abl_path}")

## 7. Comparación 9ch vs 11ch (opcional)

Ejecutar esta sección si previamente has generado tensores de 9 canales
con el notebook 01 (cambiando `N_CHANNELS=9`).

In [ ]:
# Intentar cargar tensores 9ch para comparar
tensor_9ch = os.path.join(DATA_DIR,
    f"{GENE}_tensors_9ch{'_' + SUBSET if SUBSET != 'all' else ''}.npz")

if os.path.exists(tensor_9ch) and N_CHANNELS == 11:
    d9 = np.load(tensor_9ch, allow_pickle=True)
    t9, l9 = d9["tensors"], d9["labels"]
    auc9, std9 = quick_cv(t9, l9, 9, ABL_FOLDS, ABL_EPOCHS, SEED)
    print(f"9ch AUC:  {auc9:.4f} ± {std9:.4f}")
    print(f"11ch AUC: {baseline_auc:.4f} ± {baseline_std:.4f}")
    print(f"Δ AUC:    {baseline_auc - auc9:+.4f}")
else:
    print("No se encontraron tensores 9ch para comparar.")
    print(f"Genera primero con notebook 01 usando N_CHANNELS=9")